# Lab 7: Optical Satellite Imagery — Spectral Bands and Surface Deformation

> **Colab note:** This notebook is designed to run on **Google Colab**. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amtseismo/EPS166/blob/main/notebooks/07_optical_imagery.ipynb)

## Introduction

Optical satellites record reflected sunlight across multiple wavelength bands simultaneously. By choosing which bands to assign to the red, green, and blue display channels, we can reveal surface properties — vegetation, mineralogy, moisture — that are invisible to the human eye. The same satellites, and often the same images, can also be used to measure how the ground has moved, by cross-correlating images taken before and after an earthquake.

In this lab you will:
1. Explore multispectral Sentinel-2 imagery over Crater Lake, Oregon — a site with rich spectral contrasts
2. Download and interpret a Sentinel-2 scene of your own choosing
3. Measure coseismic surface displacement from the 2019 Ridgecrest M7.1 earthquake using sub-pixel cross-correlation

## Learning objectives

By the end of this lab you will be able to:

- Load and display individual spectral bands from a multispectral satellite image
- Construct true-colour, colour-infrared (CIR), and geology composites
- Calculate and interpret NDVI
- Query and download Sentinel-2 scenes from the Copernicus Data Space API
- Apply sub-pixel cross-correlation to estimate surface displacement
- Explain why optical pixel tracking detects strike-slip motion that InSAR cannot

## Notebook outline

1. [Setup](#setup)
2. [Part I: Crater Lake — spectral exploration](#part-i-crater-lake--spectral-exploration)
3. [Part II: Your own scene](#part-ii-your-own-scene)
4. [Part III: Ridgecrest — pixel tracking](#part-iii-ridgecrest--pixel-tracking)
5. [Synthesis](#synthesis)


## Setup

Install and import required packages.


In [ ]:
%pip install -q rasterio pystac-client planetary-computer

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio
from rasterio.enums import Resampling
from skimage.registration import phase_cross_correlation
from scipy.ndimage import uniform_filter
import urllib.request, os, warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.dpi": 130, "axes.grid": False})
print("Setup complete.")


---

## Part I: Crater Lake — spectral band exploration

Crater Lake, Oregon occupies the caldera of Mount Mazama, a volcano that collapsed ~7,700 years ago. It is one of the deepest lakes in the world (~590 m) and is surrounded by dense old-growth conifer forest. The scene contains four spectrally distinct surface types:

| Surface | Spectral character |
|---|---|
| **Deep lake water** | Some penetration in blue; absorbs strongly in green, NIR, and SWIR |
| **Dense conifer forest** | Absorbs red (chlorophyll); reflects strongly in NIR (leaf cell structure) |
| **Bare volcanic rock** | Intermediate reflectance; SWIR reveals mineralogy and alteration |
| **Snow / ice** | Bright in all visible bands; dark in SWIR (water in ice absorbs it) |

We begin with a true-colour reference image so you can orient yourself in the scene, then examine individual spectral bands, and finally build false-colour composites that reveal features the eye cannot see.

### Download the Crater Lake scene

In [ ]:
# Pre-clipped Sentinel-2 L2A bands over Crater Lake (10 m resolution)
# Bands: B02 (Blue), B03 (Green), B04 (Red), B08 (NIR), B11 (SWIR-1), B12 (SWIR-2)
CRATER_BASE = (
    "https://raw.githubusercontent.com/amtseismo/EPS166/"
    "main/datasets/CraterLake/"
)
CRATER_BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]

for band in CRATER_BANDS:
    fname = f"CraterLake_{band}.tif"
    if not os.path.exists(fname):
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(CRATER_BASE + fname, fname)
print("All Crater Lake bands downloaded.")


### Load the bands

In [ ]:
def load_band(path, upsample_shape=None):
    """Load a single-band GeoTIFF. Optionally resample to upsample_shape (rows, cols)."""
    with rasterio.open(path) as src:
        if upsample_shape:
            arr = src.read(1, out_shape=upsample_shape,
                           resampling=Resampling.bilinear).astype(np.float32)
        else:
            arr = src.read(1).astype(np.float32)
        transform = src.transform
        nodata = src.nodata
    if nodata is not None:
        arr[arr == nodata] = np.nan
    return arr, transform

def pct(arr, lo=2, hi=98):
    """Percentile stretch to [0,1] for display."""
    vmin, vmax = np.nanpercentile(arr, [lo, hi])
    return np.clip((arr - vmin) / (vmax - vmin), 0, 1)

# Load 10 m bands
b02, tf = load_band("CraterLake_B02.tif")   # Blue
b03, _  = load_band("CraterLake_B03.tif")   # Green
b04, _  = load_band("CraterLake_B04.tif")   # Red
b08, _  = load_band("CraterLake_B08.tif")   # NIR

# SWIR bands are 20 m — resample to match 10 m grid
shape10 = b04.shape
b11, _  = load_band("CraterLake_B11.tif", upsample_shape=shape10)  # SWIR-1
b12, _  = load_band("CraterLake_B12.tif", upsample_shape=shape10)  # SWIR-2

print(f"Image size: {b04.shape[0]} x {b04.shape[1]} pixels  ({b04.shape[0]*10/1000:.1f} x {b04.shape[1]*10/1000:.1f} km)")

In [ ]:
def make_composite(r, g, b, lo=2, hi=98):
    """Stack three bands into an RGB display array."""
    return np.dstack([pct(r, lo, hi), pct(g, lo, hi), pct(b, lo, hi)])

### True colour — scene orientation

Before exploring individual bands, establish a reference. The true-colour composite assigns the visible bands (Red→R, Green→G, Blue→B) to produce an image close to what your eye would see from space.

As you look at the image, locate the following features — you will return to them repeatedly in the sections below:

- **The lake** — the deep blue interior
- **Dense conifer forest** — dark green slopes surrounding the caldera
- **Bare volcanic rock** — the exposed caldera rim and walls
- **Wizard Island** — a small cinder cone in the western part of the lake,   erupted *inside* the caldera after the collapse (~7,700 years ago and therefore   younger than the caldera walls); look for a dark speck of rock surrounded by   shallow water
- **Snow / ice patches** — bright white, typically on the upper rim

In [ ]:
true_color = make_composite(b04, b03, b02)

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(true_color)
ax.set_title("True colour  (B04=R, B03=G, B02=B)", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()

> **Questions — true colour:**
> 1. Locate Wizard Island. How does it appear different from the open lake water and from the forested slopes?
> 2. The lake is a vivid blue. Given that water is colourless in small quantities, what property of the lake — and which wavelengths — produce this blue appearance?
> 3. Look at the caldera rim. Can you distinguish different rock types or degrees of weathering in true colour, or does the rim appear uniform? Keep this in mind when you reach the geology composite.

### Visible bands — zoomed to Wizard Island

The deep open lake absorbs most incoming light uniformly across the visible spectrum, so a whole-scene view shows little difference between the Blue, Green, and Red bands. To see the spectral contrast that *does* exist in water, we zoom to the Wizard Island area, where shallow water near the shore, the bare volcanic rock of the island, and the deep surrounding lake are all present within a small area.

Each band is displayed in its own colour map so the spectral identity of each panel is immediately visible:

- **Blue (B02):** Penetrates clear water more deeply than other visible wavelengths;   also scattered by atmospheric particles and aerosols
- **Green (B03):** Captures peak vegetation reflectance in the visible —   the reason leaves look green
- **Red (B04):** Absorbed by chlorophyll for photosynthesis;   healthy vegetation appears dark here


In [ ]:
import pyproj

# Wizard Island centre coordinate (WGS84)
WIZ_LAT, WIZ_LON = 42.939390, -122.148470

# Reproject to the scene CRS (typically UTM Zone 10N for this part of Oregon),
# then convert to pixel row/col using the rasterio dataset index.
with rasterio.open(files["b04"]) as src:
    transformer = pyproj.Transformer.from_crs(
        "EPSG:4326", src.crs, always_xy=True
    )
    wiz_x, wiz_y = transformer.transform(WIZ_LON, WIZ_LAT)
    cy, cx = src.index(wiz_x, wiz_y)
    cy, cx = int(cy), int(cx)

pad = int(min(b04.shape) * 0.10)
rs = slice(max(0, cy - pad), cy + pad)
cs = slice(max(0, cx - pad), cx + pad)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
vis_bands = [
    (b02[rs, cs], "B02  Blue (0.49 µm)",  "Blues",  "Water penetration\natmospheric scattering"),
    (b03[rs, cs], "B03  Green (0.56 µm)", "Greens", "Peak vegetation reflectance\nin visible"),
    (b04[rs, cs], "B04  Red (0.66 µm)",   "Reds",   "Chlorophyll absorption\nfor photosynthesis"),
]
for ax, (band, title, cmap, subtitle) in zip(axes, vis_bands):
    ax.imshow(pct(band), cmap=cmap)
    ax.set_title(f"{title}\n{subtitle}", fontsize=10)
    ax.axis("off")

plt.suptitle("Visible bands — zoomed to Wizard Island (colour maps match band colour)", fontsize=12)
plt.tight_layout()
plt.show()

> **Questions — visible bands:**
> 1. Look at the shallow water immediately around Wizard Island across all three bands. In which band is the water brightest? In which is it darkest? What does this tell you about the depth to which blue vs red light penetrates water?
> 2. The rock of Wizard Island itself appears in all three bands. Does its brightness change much between bands? How does this compare to the change you see in the surrounding water?
> 3. If you can see any forested areas at the edge of the crop, compare forest brightness in B03 vs B04. Which band shows more contrast between vegetation and bare rock, and why?

### Near-Infrared: the vegetation band (B08, 0.84 µm)

The NIR band lies just beyond what the human eye can see, but it is one of the most information-rich bands in Earth observation. Two physical properties make it distinctive:

1. **Vegetation:** The internal cell structure of leaves — the spongy mesophyll layer — scatters NIR strongly upward. Healthy plants reflect ~40–50% of incident NIR, compared to ~5% of incident red.
2. **Water:** Liquid water absorbs NIR almost completely. The lake appears nearly black regardless of depth.

The dramatic contrast between these two responses is the physical basis for NDVI.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)

for ax, band, title, subtitle in zip(
    axes,
    [b04, b08],
    ["B04  Red (0.66 µm)", "B08  NIR (0.84 µm)"],
    ["Chlorophyll absorbs here —\nvegetation appears dark",
     "Leaf cell structure reflects here —\nhealthy vegetation appears bright"],
):
    ax.imshow(pct(band), cmap="gray")
    ax.set_title(f"{title}\n{subtitle}", fontsize=10)
    ax.axis("off")

plt.suptitle("Red vs Near-Infrared: the contrast that underlies NDVI", fontsize=12)
plt.tight_layout()
plt.show()

> **Questions — NIR:**
> 1. How does the brightness of the forested slopes change between B04 and B08? What physical property of leaves explains this?
> 2. The lake is one of the clearest and deepest in the world. Does NIR penetrate it at all, or does it appear uniformly dark? What does this imply about using NIR for bathymetric mapping?
> 3. Examine the bare volcanic rock on the caldera rim. How much does its brightness change between B04 and B08 compared to the change in the forest? Why is the forest's response so much more dramatic?

### Shortwave-Infrared: mineralogy and moisture (B11, B12)

The SWIR bands (B11 at 1.61 µm and B12 at 2.19 µm) carry information invisible to the human eye but physically distinctive:

- **Ice and snow:** Water molecules in ice crystals absorb SWIR strongly. Snow that appears bright white in visible looks dark in SWIR — a key way to distinguish fresh snow from bright rock or sand.
- **Leaf water content:** Liquid water in plant leaves absorbs SWIR. Drier or stressed vegetation appears brighter in SWIR than well-watered vegetation.
- **Rock mineralogy:** Clay minerals and hydrothermal alteration products have diagnostic absorption features in the SWIR. The volcanic walls of the caldera may show subtle variation reflecting different rock types or weathering stages.

*Note: B11 and B12 are native 20 m resolution — resampled here to match the 10 m visible/NIR grid.*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)

swir_bands = [
    (b08, "B08  NIR (0.84 µm)", "(reference — vegetation bright,\nwater dark)"),
    (b11, "B11  SWIR-1 (1.61 µm)", "Leaf water content;\nsnow/ice appears dark"),
    (b12, "B12  SWIR-2 (2.19 µm)", "Clay minerals and hydrothermal\nalteration detectable"),
]
for ax, (band, title, subtitle) in zip(axes, swir_bands):
    ax.imshow(pct(band), cmap="gray")
    ax.set_title(f"{title}\n{subtitle}", fontsize=10)
    ax.axis("off")

plt.suptitle("NIR vs SWIR: moisture, ice, and mineralogy", fontsize=12)
plt.tight_layout()
plt.show()

> **Questions — SWIR:**
> 1. The most striking difference in B11 and B12 compared to NIR is how snow and ice appear. Describe what you observe and explain why ice absorbs SWIR so strongly.
> 2. Compare the forest brightness in B08 (NIR) vs B11 (SWIR-1). Does the forest appear brighter or darker in SWIR? What does this tell you about the water content of the trees?
> 3. Look carefully at the caldera walls in B12. Can you see colour or brightness variation that is not obvious in the visible bands? What geological process might produce that variation?

### False-colour composites

You have now seen how each wavelength region responds to the scene. We build two more composites that assign non-visible bands to display channels, revealing features that are invisible in the true-colour image you examined at the start.

| Composite | Band → Red | Band → Green | Band → Blue | Key use |
|---|---|---|---|---|
| **Colour Infrared (CIR)** | B08 NIR | B04 Red | B03 Green | Vegetation health, water depth |
| **Geology** | B12 SWIR-2 | B11 SWIR-1 | B04 Red | Mineralogy, discriminates snow from rock |

#### Colour Infrared / CIR (B08→R, B04→G, B03→B)

Replaces the blue display channel with red, green with red, and red with NIR. This "shifts" the display one channel toward longer wavelengths, making NIR information appear as red. Widely used in ecology and agriculture; originally designed to mimic wartime colour-infrared aerial film.

In [ ]:
cir = make_composite(b08, b04, b03)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(cir)
ax.set_title("Colour Infrared / CIR  (B08=R, B04=G, B03=B)", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()

> **Questions — CIR:**
> 1. What colour is the dense conifer forest in CIR? Explain this in terms of which bands are assigned to which display channels and what reflectances you measured in the individual band exploration.
> 2. How does the lake appear in CIR compared to true colour? Is there any colour variation across the lake (e.g. shallow vs deep areas near the shore)?
> 3. Wizard Island is the small volcanic cone near the western edge of the lake. How does it appear relative to the surrounding forest, and what does that tell you about its surface cover?
> 4. Where does snow appear in the CIR image? Is it easy to distinguish snow from bare rock in this composite?

#### Geology composite (B12→R, B11→G, B04→B)

Replaces the visible display channels with SWIR bands sensitive to mineralogy. Because ice absorbs SWIR strongly, snow that appears white in true colour appears dark here — making it easy to separate snow from bright rock. Clay minerals and hydrothermal alteration zones appear in distinct colours.

In [ ]:
geology = make_composite(b12, b11, b04)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(geology)
ax.set_title("Geology  (B12=R, B11=G, B04=B)", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()

> **Questions — geology:**
> 1. What colour does snow appear in the geology composite? Explain this using what you observed in B11 and B12 during the SWIR section.
> 2. Look at the caldera walls. Do you see colour variation that was absent in true colour? What might different colours indicate geologically?
> 3. How does the forest appear in the geology composite compared to CIR? Explain the change in terms of what B11 and B12 measure for vegetation.
> 4. If you were mapping the distribution of hydrothermal alteration on a volcano, which composite would you choose and why?

### NDVI

Instead of combining bands for display, we can compute a mathematical index that directly exploits the contrast we saw between B04 and B08. NDVI encodes, in a single number, how much more strongly a surface reflects NIR than red — the signature of healthy vegetation.

$$\text{NDVI} = \frac{\rho_{\text{NIR}} - \rho_{\text{red}}}{\rho_{\text{NIR}} + \rho_{\text{red}}}$$

In [ ]:
with np.errstate(divide="ignore", invalid="ignore"):
    ndvi = (b08 - b04) / (b08 + b04)
ndvi = np.where(np.isfinite(ndvi), ndvi, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(true_color)
axes[0].set_title("True colour", fontsize=11)
axes[0].axis("off")

im = axes[1].imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[1].set_title("NDVI", fontsize=11)
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], shrink=0.7, label="NDVI")

plt.suptitle("Crater Lake — true colour vs NDVI", fontsize=12)
plt.tight_layout()
plt.show()

print(f"NDVI range: {np.nanmin(ndvi):.2f} to {np.nanmax(ndvi):.2f}")
print(f"Mean NDVI over forested area: ~{np.nanmean(ndvi[ndvi > 0.3]):.2f}")


> **Questions — NDVI:**
> 1. What NDVI value does the lake surface have, and why? What physical interpretation does a negative NDVI carry?
> 2. The caldera rim is bare volcanic rock with very little vegetation. Where do you expect it to fall on the NDVI scale, and does the map confirm this?
> 3. Is there any spatial variation in NDVI within the forested area? What might cause patches of lower NDVI within an otherwise dense forest?
> 4. Wizard Island is a cinder cone that erupted *inside* the caldera after the Mount Mazama collapse (~7,700 years ago), making it significantly younger than the caldera walls. Compare the NDVI of Wizard Island to the surrounding caldera rim. Do they differ? If so, explain the difference in terms of surface age, soil development, and vegetation establishment on volcanic terrain.

---

## Part II: Your own scene

Now you will download a Sentinel-2 scene of your own choice from the Copernicus Data Space Ecosystem using its STAC API. Choose any location that interests you geologically — a volcano, a fault zone, a river delta, a glaciated mountain range, a mining region.

### How the Copernicus STAC API works

The Copernicus Data Space provides a public STAC (SpatioTemporal Asset Catalog) endpoint that you can query without credentials. Each result is a Sentinel-2 granule with direct download links for individual bands.

The steps are:
1. Define a bounding box and date range for your area of interest
2. Search for available scenes with low cloud cover
3. Pick one result and download the bands you need
4. Display composites as you did for Crater Lake


In [ ]:
import pystac_client
import planetary_computer
import requests

# ── Define your area of interest ──────────────────────────────────────────────
# Replace these coordinates with your chosen location
# Format: [west, south, east, north] in decimal degrees
MY_BBOX = [-122.2, 42.80, -121.9, 43.05]   # ← edit this

# Date range to search (YYYY-MM-DD/YYYY-MM-DD)
MY_DATES = "2023-07-01/2023-09-30"          # ← edit this

# Maximum cloud cover percentage
MAX_CLOUD = 10

# ── Search the catalog ────────────────────────────────────────────────────────
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=MY_BBOX,
    datetime=MY_DATES,
    query={"eo:cloud_cover": {"lt": MAX_CLOUD}},
    sortby="+properties.datetime",
)

items = list(search.items())
print(f"Found {len(items)} scenes with < {MAX_CLOUD}% cloud cover")
for i, item in enumerate(items[:5]):
    print(f"  [{i}]  {item.datetime.date()}   cloud: {item.properties['eo:cloud_cover']:.0f}%   id: {item.id}")


In [ ]:
# ── Download bands from your chosen scene ────────────────────────────────────
# Change this index to select a different scene from the list above
SCENE_IDX = 0
item = items[SCENE_IDX]
print(f"Using scene: {item.datetime.date()}  (cloud cover {item.properties['eo:cloud_cover']:.0f}%)")

def download_band_from_item(item, band_name, out_path):
    """Download one band from a Planetary Computer STAC item."""
    if os.path.exists(out_path):
        return
    url = item.assets[band_name].href
    print(f"  Downloading {band_name}...")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)

for band in ["B02", "B03", "B04", "B08"]:
    download_band_from_item(item, band, f"my_scene_{band}.tif")
print("Download complete.")


In [ ]:
# ── Load and display your scene ───────────────────────────────────────────────
my_b02, _ = load_band("my_scene_B02.tif")
my_b03, _ = load_band("my_scene_B03.tif")
my_b04, _ = load_band("my_scene_B04.tif")
my_b08, _ = load_band("my_scene_B08.tif")

my_true = make_composite(my_b04, my_b03, my_b02)
my_cir  = make_composite(my_b08, my_b04, my_b03)

with np.errstate(divide="ignore", invalid="ignore"):
    my_ndvi = (my_b08 - my_b04) / (my_b08 + my_b04)
my_ndvi = np.where(np.isfinite(my_ndvi), my_ndvi, np.nan)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, img, title in zip(axes,
    [my_true, my_cir, my_ndvi],
    ["True colour", "Colour Infrared (CIR)", "NDVI"]):
    if img.ndim == 2:
        im = ax.imshow(img, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
        plt.colorbar(im, ax=ax, shrink=0.6)
    else:
        ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.suptitle(f"Your scene — {item.datetime.date()}", fontsize=12)
plt.tight_layout()
plt.show()


> **Describe your scene:**
> 1. Where did you choose and why? What geological or surface features did you expect to see?
> 2. What does the true-colour image show that matches your expectations? What surprised you?
> 3. How does the CIR composite change your interpretation of the scene compared to true colour?
> 4. Describe the NDVI pattern. Does it make sense given what you know about the area?


---

## Part III: Ridgecrest — pixel tracking

On 5 July 2019, a M7.1 earthquake struck the Ridgecrest area of the Mojave Desert, California. The right-lateral rupture on the Airport Lake fault zone produced ~4–5 m of peak surface slip along a NW-trending trace. The desert surface — bare rock and alluvial fans with no vegetation — is ideal for sub-pixel cross-correlation: the texture is stable between images and not affected by seasonal changes.

In this section you will:
- download pre- and post-event Sentinel-2 NIR (Band 8) images
- cross-correlate image chips to estimate surface displacement
- extract a fault-perpendicular profile and estimate total slip
- calculate how insensitive InSAR would be to this strike-slip signal


### Download the imagery

Both scenes are pre-clipped Sentinel-2 Band 8 (NIR, 10 m resolution) GeoTIFFs covering a ~50 × 50 km window centered on the Ridgecrest rupture zone. They are hosted on the course GitHub repository so no API credentials are needed.


In [ ]:
BASE_URL = (
    "https://github.com/amtseismo/EPS166/"
    "raw/main/datasets/"
)

files = {
    "pre":  "ridgecrest_pre_B08.tif",
    "post": "ridgecrest_post_B08.tif",
}

for key, fname in files.items():
    if not os.path.exists(fname):
        print(f"Downloading {fname} ...")
        urllib.request.urlretrieve(BASE_URL + fname, fname)
    else:
        print(f"{fname} already present.")

print("\nDone.")

### Load the images

Read each GeoTIFF into a NumPy array and extract the geotransform so we can plot in real-world coordinates (UTM meters).


In [ ]:
def load_band_geo(path):
    """
    Load a single-band GeoTIFF.

    Parameters
    ----------
    path : str
        File path to the GeoTIFF.

    Returns
    -------
    arr : numpy.ndarray  (rows, cols), float32
        Image data, nodata pixels set to NaN.
    transform : affine.Affine
        Affine geotransform (pixel → map coordinates).
    crs : rasterio.crs.CRS
        Coordinate reference system.
    extent : list
        [left, right, bottom, top] in map units for imshow.
    """
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        left, bottom, right, top = src.bounds
    if nodata is not None:
        arr[arr == nodata] = np.nan
    extent = [left, right, bottom, top]
    return arr, transform, crs, extent


pre,  tf_pre,  crs, extent = load_band(files["pre"])
post, tf_post, _,   _      = load_band(files["post"])

pixel_size_m = abs(tf_pre.a)   # pixel size in metres (should be 10 m)

print(f"Image size  : {pre.shape[0]} rows × {pre.shape[1]} cols")
print(f"Pixel size  : {pixel_size_m:.0f} m")
print(f"CRS         : {crs.to_string()}")
print(f"Extent (km) : W={extent[0]/1e3:.1f}  E={extent[1]/1e3:.1f}  "
      f"S={extent[2]/1e3:.1f}  N={extent[3]/1e3:.1f}")

### Visualise pre- and post-event images

Before any processing, look at the data. The Mojave Desert has excellent optical coherence: sparse vegetation, stable sandy and rocky surfaces, and low cloud cover. These properties make it ideal for pixel tracking.

The surface rupture of the M7.1 mainshock is visible in the post-event image as a linear brightness change caused by freshly disturbed desert pavement.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)

for ax, img, title in zip(
    axes,
    [pre, post],
    ["Pre-event  (2019-06-24)", "Post-event (2019-07-14)"],
):
    ax.imshow(
        pct(img),
        cmap="gray",
        extent=[e / 1e3 for e in extent],
        origin="upper",
        interpolation="none",
    )
    ax.set_title(title)
    ax.set_xlabel("Easting (km)")

axes[0].set_ylabel("Northing (km)")
fig.suptitle(
    "Sentinel-2 Band 8 (NIR, 10 m) — Ridgecrest, CA",
    fontsize=13,
)
plt.tight_layout()
plt.show()

print("\nCan you see the M7.1 surface rupture in the post-event image?")
print("It runs roughly NW–SE through the centre of the scene.")

> **Question 2.1:** Describe any visible differences between the pre- and post-event images. Where do you see evidence of surface change? What direction does the rupture trend?

> **Question 2.2:** Why is Band 8 (near-infrared) preferred for pixel tracking over desert terrain compared to a visible band?

### Why NIR for desert pixel tracking?

The near-infrared band (Band 8, 0.84–0.88 µm) is preferred for pixel tracking in desert environments for two reasons:

1. **Vegetation contrast.** Sparse desert shrubs reflect NIR strongly relative to bare rock and sand. This creates stable high-contrast texture that the cross-correlation can lock onto.
2. **Atmospheric scattering.** Longer wavelengths scatter less in the atmosphere than visible blue or green light, giving a cleaner surface signal.

For a co-seismic application like this one, both images are cloud-free Mojave summer scenes, so atmospheric differences are minimal. The main advantage here is the texture.


### Coregistration check

Pixel tracking measures the **difference** in position of surface features between two images. Any apparent displacement that exists everywhere in the scene — not just near the fault — is likely a residual misregistration or orbital error, not real ground motion.

We check coregistration by running the same cross-correlation on a **stable reference region** far from the fault. If the two images are well aligned, the recovered offset should be near zero everywhere in the reference region.


In [ ]:
# --- Define a stable reference region far from the rupture ---
# Adjust row/col indices if needed once you have seen the images.
# This corner region should be well away from the fault trace.
REF_ROWS = slice(50, 250)
REF_COLS = slice(50, 250)

pre_ref  = pre[REF_ROWS, REF_COLS]
post_ref = post[REF_ROWS, REF_COLS]

shift_ref, error_ref, _ = phase_cross_correlation(
    pre_ref, post_ref, upsample_factor=100
)

az_bias_m = -shift_ref[0] * pixel_size_m   # row shift → northward metres
rg_bias_m = -shift_ref[1] * pixel_size_m   # col shift → eastward metres

print("Coregistration check (reference region)")
print("-" * 40)
print(f"Azimuth (N–S) offset : {az_bias_m:+.3f} m")
print(f"Range   (E–W) offset : {rg_bias_m:+.3f} m")
print()
if abs(az_bias_m) < 1.0 and abs(rg_bias_m) < 1.0:
    print("✓  Residual offset < 1 m in both directions. Images are well aligned.")
else:
    print("⚠  Residual offset > 1 m — consider applying a bulk correction.")
    print(f"   Subtracting az_bias={az_bias_m:.3f} m, rg_bias={rg_bias_m:.3f} m")
    print("   from all profile measurements below.")

> **Question 3.1:** If the reference-region offset were 2 m rather than near-zero, what would that imply about the pre- and post-event images? How would you correct for it?

> **Question 3.2:** Would you expect better or worse coregistration between two images from the same Sentinel-2 pass direction (e.g. both descending) compared to one ascending and one descending? Why?


### Sub-pixel cross-correlation

### How pixel tracking works

We divide the image into small overlapping **chips** (windows). For each chip we:

1. Extract the same-location chip from the pre- and post-event images.
2. Compute the cross-correlation between the two chips.
3. Find the peak of the correlation — its offset from centre gives the displacement of that chip.

We use `phase_cross_correlation` from `scikit-image`, which works in the Fourier domain and can resolve displacements to a small fraction of a pixel using the `upsample_factor` parameter.

**Sign convention:**
- Positive azimuth offset → the surface moved **northward** between the two images.
- Positive range offset → the surface moved **eastward**.

For a right-lateral rupture on a NW-trending fault:
- The NE block (right side looking NW along the fault) should move **SE** (negative azimuth, positive range).
- The SW block should move **NW** (positive azimuth, negative range).


In [ ]:
def correlate_chip(pre_img, post_img, row, col, chip=64, upsample=20):
    """
    Estimate sub-pixel displacement for one image chip.

    Parameters
    ----------
    pre_img, post_img : numpy.ndarray
        Full pre- and post-event images.
    row, col : int
        Centre pixel of the chip.
    chip : int
        Full side length of the chip in pixels (half = chip // 2 is used internally).
    upsample : int
        Upsampling factor for sub-pixel precision.

    Returns
    -------
    az_m, rg_m : float
        Azimuth (N+) and range (E+) displacement in metres.
        Returns (nan, nan) if the chip contains NaN or has no variance.
    """
    half = chip // 2
    r0, r1 = row - half, row + half
    c0, c1 = col - half, col + half

    if r0 < 0 or r1 > pre_img.shape[0] or c0 < 0 or c1 > pre_img.shape[1]:
        return np.nan, np.nan

    pre_chip  = pre_img[r0:r1, c0:c1]
    post_chip = post_img[r0:r1, c0:c1]

    if np.any(np.isnan(pre_chip)) or np.any(np.isnan(post_chip)):
        return np.nan, np.nan
    if pre_chip.std() < 1e-6 or post_chip.std() < 1e-6:
        return np.nan, np.nan

    shift, _, _ = phase_cross_correlation(
        pre_chip, post_chip, upsample_factor=upsample
    )
    az_m = -shift[0] * pixel_size_m   # row → N (sign flip: +row = south)
    rg_m = -shift[1] * pixel_size_m   # col → E
    return az_m, rg_m

### Displacement profile across the fault

We extract a single fault-perpendicular profile — a row of chips sampled at regular intervals across the rupture zone. The fault trends NW–SE at roughly 315°, so a W–E profile is approximately perpendicular to it.

The profile row is set near the centre of the scene where the M7.1 rupture is best expressed. Adjust `PROFILE_ROW` if needed after inspecting the images in Part II.


In [ ]:
# --- Profile parameters ---
CHIP_SIZE  = 64    # chip side length in pixels (640 m)
STRIDE     = 20    # spacing between chip centres in pixels (200 m)
UPSAMPLE   = 20    # sub-pixel precision factor (0.5 m at 10 m pixels)

# Row through the middle of the scene — adjust if the rupture is off-centre
PROFILE_ROW = pre.shape[0] // 2

half = CHIP_SIZE // 2
col_centres = range(half, pre.shape[1] - half, STRIDE)

az_offsets_m = []
rg_offsets_m = []

for col in col_centres:
    az, rg = correlate_chip(
        pre, post, PROFILE_ROW, col,
        chip=CHIP_SIZE, upsample=UPSAMPLE
    )
    az_offsets_m.append(az)
    rg_offsets_m.append(rg)

col_centres  = np.array(list(col_centres))
az_offsets_m = np.array(az_offsets_m)
rg_offsets_m = np.array(rg_offsets_m)

# Convert pixel column index to easting (km) for plotting
easting_km = (extent[0] + col_centres * pixel_size_m) / 1e3

print(f"Profile row        : {PROFILE_ROW}  "
      f"(northing ≈ {(extent[3] - PROFILE_ROW * pixel_size_m)/1e3:.1f} km)")
print(f"Number of chips    : {len(col_centres)}")
print(f"Valid measurements : {np.sum(~np.isnan(az_offsets_m))}")
print(f"Azimuth range      : {np.nanmin(az_offsets_m):.1f} to "
      f"{np.nanmax(az_offsets_m):.1f} m")

In [ ]:
# --- Plot the displacement profile ---
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# Subtract any residual bulk offset estimated from the reference region
az_corrected = az_offsets_m - az_bias_m
rg_corrected = rg_offsets_m - rg_bias_m

axes[0].plot(easting_km, az_corrected, "C0.", ms=4, label="Azimuth (N+)")
axes[0].axhline(0, color="k", lw=0.8, ls="--")
axes[0].set_ylabel("Displacement (m)")
axes[0].set_title("Azimuth offset  (northward = positive)")
axes[0].legend()

axes[1].plot(easting_km, rg_corrected, "C1.", ms=4, label="Range (E+)")
axes[1].axhline(0, color="k", lw=0.8, ls="--")
axes[1].set_ylabel("Displacement (m)")
axes[1].set_xlabel("Easting (km)")
axes[1].set_title("Range offset  (eastward = positive)")
axes[1].legend()

fig.suptitle(
    f"Fault-perpendicular displacement profile  (row {PROFILE_ROW})",
    fontsize=13,
)
plt.tight_layout()
plt.show()

> **Question 5.1:** Describe the shape of the azimuth offset profile. Is it consistent with right-lateral slip on a NW-trending fault? Explain which side moved which direction.

> **Question 5.2:** The range (E–W) profile should also show a step. Is the step in the same direction as you would expect for right-lateral motion? What does the combination of azimuth and range offsets tell you about the slip vector?

> **Question 5.3:** Are there any chips far from the fault with anomalously large offsets? What might cause those outliers?


### Estimating total slip

The total surface slip is the **step** in the displacement profile across the fault — the difference between the mean offset on the NE block and the mean offset on the SW block.

We identify the fault location by finding where the azimuth gradient is steepest, then compute block means on either side.


In [ ]:
# --- Locate the fault: maximum gradient in the azimuth profile ---
valid = ~np.isnan(az_corrected)
az_valid  = az_corrected[valid]
east_valid = easting_km[valid]

# Smooth before differencing to reduce noise
smooth_az = uniform_filter(az_valid, size=5)
gradient  = np.abs(np.gradient(smooth_az))
fault_idx = np.argmax(gradient)
fault_km  = east_valid[fault_idx]

print(f"Fault located at easting ≈ {fault_km:.1f} km")

# --- Block means 10 km either side of the fault ---
BLOCK_KM = 10.0
sw_mask = (east_valid > fault_km - 2*BLOCK_KM) & (east_valid < fault_km - 2.0)
ne_mask = (east_valid > fault_km + 2.0) & (east_valid < fault_km + 2*BLOCK_KM)

az_sw = np.nanmean(az_valid[sw_mask])
az_ne = np.nanmean(az_valid[ne_mask])
az_step = az_ne - az_sw

rg_valid = rg_corrected[valid]
rg_sw = np.nanmean(rg_valid[sw_mask])
rg_ne = np.nanmean(rg_valid[ne_mask])
rg_step = rg_ne - rg_sw

total_slip = np.sqrt(az_step**2 + rg_step**2)

print(f"\nSlip components")
print("-" * 35)
print(f"Azimuth step (NE − SW) : {az_step:+.2f} m")
print(f"Range   step (NE − SW) : {rg_step:+.2f} m")
print(f"Total horizontal slip  : {total_slip:.2f} m")

# --- Annotate the profile plot ---
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(east_valid, az_valid, "C0.", ms=4, alpha=0.6)
ax.axvline(fault_km, color="r", lw=1.5, ls="--", label=f"Fault @ {fault_km:.1f} km")
ax.axhline(az_sw, xmin=0, xmax=(fault_km - east_valid[0]) / (east_valid[-1] - east_valid[0]),
           color="C2", lw=2, label=f"SW block mean = {az_sw:.2f} m")
ax.axhline(az_ne,
           xmin=(fault_km - east_valid[0]) / (east_valid[-1] - east_valid[0]),
           color="C3", lw=2, label=f"NE block mean = {az_ne:.2f} m")
ax.annotate(
    f"Step = {az_step:+.2f} m",
    xy=(fault_km, (az_sw + az_ne) / 2),
    xytext=(fault_km + 3, (az_sw + az_ne) / 2),
    arrowprops=dict(arrowstyle="->"),
    fontsize=10,
)
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Azimuth offset (m)")
ax.set_title("Azimuth displacement profile with fault location and block means")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

> **Question 6.1:** The USGS and Caltech field teams measured peak surface slip of ~4–5 m on the M7.1 rupture. How does your estimate compare? List at least two reasons the optical pixel-tracking estimate might differ from field measurements.

> **Question 6.2:** The azimuth step and range step together define a 2D slip vector. Compute the azimuth of that vector (degrees clockwise from north) and comment on whether it is consistent with right-lateral slip on a NW-trending fault.

$$
\text{slip azimuth} = \arctan\!\left(\frac{\Delta_{\text{range}}}{\Delta_{\text{azimuth}}}\right)
$$


*Hint: use `np.degrees(np.arctan2(rg_step, az_step))` rather than plain `arctan` — `atan2` handles all four quadrants correctly by considering the signs of both components.*

### Why InSAR misses this signal

Sentinel-1 acquires SAR imagery from a descending orbit with an incidence angle of ~38° and a look azimuth of ~349° (nearly north, slightly west). The Ridgecrest M7.1 fault trends at approximately **315°** (NW–SE).

InSAR measures displacement in the **line-of-sight (LOS)** direction — a unit vector pointing from the ground to the satellite:

$$
\hat{\mathbf{l}} = \begin{pmatrix}
\sin\theta \sin\phi \\
\sin\theta \cos\phi \\
\cos\theta
\end{pmatrix}
$$

where $\theta$ is the incidence angle and $\phi$ is the look azimuth (from north, clockwise).

The fault-parallel unit vector (along the fault strike, in the direction of slip) is:

$$
\hat{\mathbf{f}} = \begin{pmatrix}
\sin\alpha \\
\cos\alpha \\
0
\end{pmatrix}
$$

where $\alpha$ is the fault strike. The InSAR sensitivity to fault-parallel horizontal slip is $\hat{\mathbf{l}} \cdot \hat{\mathbf{f}}$.


In [ ]:
# --- Sentinel-1 descending geometry ---
theta_deg = 38.0    # incidence angle
phi_deg   = 349.0   # look azimuth (from north, clockwise; descending right-looking)
alpha_deg = 315.0   # fault strike (NW–SE right-lateral)

theta = np.deg2rad(theta_deg)
phi   = np.deg2rad(phi_deg)
alpha = np.deg2rad(alpha_deg)

# LOS unit vector (E, N, Up)
l_hat = np.array([
    np.sin(theta) * np.sin(phi),
    np.sin(theta) * np.cos(phi),
    np.cos(theta),
])

# Fault-parallel unit vector (E, N, Up) — horizontal slip vector
f_hat = np.array([np.sin(alpha), np.cos(alpha), 0.0])

sensitivity = np.dot(l_hat, f_hat)

# For comparison: sensitivity to vertical
v_hat = np.array([0.0, 0.0, 1.0])
sensitivity_vert = np.dot(l_hat, v_hat)

print("Sentinel-1 descending LOS geometry")
print("-" * 45)
print(f"Incidence angle        : {theta_deg}°")
print(f"Look azimuth           : {phi_deg}°  (from N, CW)")
print(f"Fault strike           : {alpha_deg}°")
print()
print(f"LOS sensitivity to fault-parallel slip : {sensitivity:.3f}")
print(f"LOS sensitivity to vertical            : {sensitivity_vert:.3f}")
print()
print(f"If total slip = {total_slip:.2f} m (your estimate):")
print(f"  Expected InSAR LOS signal  = {abs(sensitivity * total_slip):.2f} m")
print(f"  Optical azimuth signal     ≈ {abs(az_step):.2f} m  (measured)")

In [ ]:
# --- Sensitivity as a function of fault strike ---
strikes = np.linspace(0, 360, 361)
sens_vals = [
    np.dot(l_hat, np.array([np.sin(np.deg2rad(s)), np.cos(np.deg2rad(s)), 0.0]))
    for s in strikes
]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(strikes, sens_vals, "C0", lw=1.8)
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.axvline(alpha_deg, color="r", lw=1.5, ls="--",
           label=f"Ridgecrest fault (strike={alpha_deg}°) → sensitivity={sensitivity:.3f}")
ax.fill_between(strikes, sens_vals, 0,
                where=np.abs(sens_vals) < 0.1,
                alpha=0.2, color="orange",
                label="|sensitivity| < 0.1  (InSAR blind zone)")
ax.set_xlabel("Fault strike (° from N, CW)")
ax.set_ylabel("LOS sensitivity to fault-parallel slip")
ax.set_title("Sentinel-1 descending LOS sensitivity to horizontal strike-slip")
ax.legend(fontsize=9)
ax.set_xlim(0, 360)
plt.tight_layout()
plt.show()

> **Question 7.1:** Based on the sensitivity calculation, how large would the InSAR LOS signal be for the M7.1 rupture? Is this detectable? What fraction of the true slip would InSAR recover?

> **Question 7.2:** Look at the sensitivity curve. For what fault orientations is InSAR *most* sensitive to horizontal strike-slip motion? What kind of faults are those?

> **Question 7.3:** How could you combine ascending and descending InSAR tracks, together with optical pixel tracking, to fully resolve the 3D displacement vector?


### Extension — 2D displacement map

If you have time, compute the azimuth offset on a regular grid across the entire scene to produce a 2D map. This requires running the cross-correlation for every grid node — it will take a few minutes on Colab.

The output should show the classic "butterfly" pattern of a strike-slip earthquake: two lobes of opposite-sign displacement separated by the fault trace.


In [ ]:
# --- 2D displacement map (coarser grid to keep runtime reasonable) ---
GRID_STRIDE = 40     # pixels between grid nodes (400 m spacing)
CHIP_2D     = 64
UPSAMPLE_2D = 10

half2 = CHIP_2D // 2
row_nodes = range(half2, pre.shape[0] - half2, GRID_STRIDE)
col_nodes = range(half2, pre.shape[1] - half2, GRID_STRIDE)

n_rows = len(list(row_nodes))
n_cols = len(list(col_nodes))

az_map = np.full((n_rows, n_cols), np.nan)
rg_map = np.full((n_rows, n_cols), np.nan)

print(f"Computing {n_rows} × {n_cols} = {n_rows*n_cols} chips ...")

for i, row in enumerate(row_nodes):
    for j, col in enumerate(col_nodes):
        az, rg = correlate_chip(
            pre, post, row, col,
            chip=CHIP_2D, upsample=UPSAMPLE_2D
        )
        az_map[i, j] = az - az_bias_m
        rg_map[i, j] = rg - rg_bias_m
    if i % 5 == 0:
        print(f"  row {i+1}/{n_rows} done")

print("Done.")

In [ ]:
# --- Plot the 2D azimuth offset map ---
clim = np.nanpercentile(np.abs(az_map), 95)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, data, title in zip(
    axes,
    [az_map, rg_map],
    ["Azimuth offset  (northward = positive, m)",
     "Range offset  (eastward = positive, m)"],
):
    im = ax.imshow(
        data,
        cmap="RdBu_r",
        vmin=-clim, vmax=clim,
        extent=[e / 1e3 for e in extent],
        origin="upper",
        aspect="equal",
        interpolation="nearest",
    )
    plt.colorbar(im, ax=ax, label="m", shrink=0.7)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Easting (km)")

axes[0].set_ylabel("Northing (km)")
fig.suptitle(
    "2D pixel-tracking displacement  —  Ridgecrest M7.1",
    fontsize=13,
)
plt.tight_layout()
plt.show()

> **Extension question:** Describe the spatial pattern of the azimuth offset map. Does it show the expected pattern for a right-lateral rupture? Where is the signal noisiest, and why might that be?


---
## Synthesis

> **In a short paragraph, summarise what you measured and what it tells you about the Ridgecrest M7.1 earthquake. Your response should:**
> 1. state your estimated total surface slip and compare it to published field measurements;
> 2. using the azimuth and range steps from your profile, compute the slip vector azimuth (using `atan2`) and comment on whether it is consistent with right-lateral slip on a NW-trending fault;
> 3. explain quantitatively why Sentinel-1 descending InSAR has low sensitivity to this rupture;
> 4. describe one situation where InSAR would be more useful than optical pixel tracking, and one where optical would be preferred.

**References**
- Barnhart, W. D., et al. (2019). Geodetic constraints on the 2019 Ridgecrest earthquake sequence. *Seismological Research Letters*, 91(4), 2056–2066. https://doi.org/10.1785/0220190322
- Leprince, S., et al. (2007). Automatic and precise orthorectification, coregistration, and subpixel correlation of satellite images. *IEEE TGRS*, 45(6), 1529–1558.
- Liu, C., et al. (2019). Three‐dimensional surface displacements of the 2019 Ridgecrest, California, earthquake sequence. *Geophysical Research Letters*, 46(22), 12764–12772.
- Sentinel-2 imagery: Copernicus Data Space Ecosystem, European Space Agency.


---

## Summary

- Optical pixel tracking measures surface displacement by cross-correlating image chips from pre- and post-event satellite images at sub-pixel precision.
- A coregistration check on a stable off-fault region is essential before interpreting any displacement signal.
- For the Ridgecrest M7.1, the azimuth offset profile shows a clear step across the NW-trending rupture consistent with right-lateral slip.
- The total horizontal slip estimated from azimuth and range offsets is comparable to published field measurements.
- Sentinel-1 descending InSAR has low sensitivity (~0) to fault-parallel horizontal motion on NW-trending faults because the LOS is nearly perpendicular to the slip direction.
- Optical pixel tracking in the image azimuth direction is sensitive to exactly the component InSAR misses, making the two techniques complementary.
- The 2D displacement map reveals the spatial pattern of co-seismic deformation across the entire scene.

This lab used Sentinel-2 Band 8 imagery processed for EPS 166 from the Copernicus Data Space Ecosystem. The pixel-tracking approach follows Leprince et al. (2007).
